# 📘 Auto Loader & File-Based Ingestion
## Automatic File Discovery, Schema Inference & Exactly-Once Processing

**What you'll learn:**
- Auto Loader fundamentals (cloudFiles format)
- File detection modes (directory listing vs file notification)
- Schema inference and evolution
- Handling different file formats (JSON, CSV, Parquet, Avro)
- Rescue data column (handling corrupt records)
- Checkpointing and exactly-once semantics
- Production patterns for file ingestion
- Auto Loader vs manual spark.read vs Kafka

**Prerequisites:** Notebooks 04 (Medallion), 24 (Streaming)
**Study Order:** 9
**Reference:** https://docs.databricks.com/ingestion/cloud-object-storage/auto-loader/

---

---
# 📗 Section 1: Why Auto Loader?

## The File Ingestion Problem

New files arrive continuously in cloud storage (S3, ADLS, GCS).
How do you process them reliably?

| Approach | How | Problems |
|----------|-----|----------|
| **Manual spark.read** | Read entire directory | Reprocesses old files, no tracking |
| **Custom tracking** | Track processed files in a table | Complex, error-prone, race conditions |
| **Auto Loader** | Automatic discovery + tracking | ✅ Simple, reliable, exactly-once |

## Auto Loader Architecture

```
Cloud Storage (S3/ADLS/GCS)          Auto Loader              Delta Table
┌─────────────────────────┐    ┌──────────────────┐    ┌──────────────────┐
│ /raw/orders/            │    │                  │    │                  │
│   2024-03-15_001.json   │───▶│ Discovers new    │───▶│ bronze.orders    │
│   2024-03-15_002.json   │    │ files            │    │                  │
│   2024-03-16_001.json ← │    │ Tracks processed │    │ (append-only)    │
│   (NEW!)                │    │ Handles schema   │    │                  │
└─────────────────────────┘    └──────────────────┘    └──────────────────┘
```

## Core API

```python
# Basic Auto Loader pattern
df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/checkpoints/orders/schema")
    .option("cloudFiles.inferColumnTypes", "true")
    .load("/raw/orders/"))

# Write to Delta (Bronze layer)
(df.writeStream
    .format("delta")
    .option("checkpointLocation", "/checkpoints/orders/")
    .trigger(availableNow=True)  # Process all available, then stop
    .outputMode("append")
    .toTable("bronze.orders"))
```

In [ ]:
# Auto Loader demo (works on Community Edition with directory listing mode)
# First, create some sample files to ingest

# Create a directory with JSON files
import json, os

# Simulate file drops
sample_dir = "/tmp/auto_loader_demo"
os.makedirs(sample_dir, exist_ok=True)

# Write sample JSON files
files_data = [
    [{"order_id": 1, "customer": "Alice", "amount": 100.0, "date": "2024-03-15"},
     {"order_id": 2, "customer": "Bob", "amount": 200.0, "date": "2024-03-15"}],
    [{"order_id": 3, "customer": "Charlie", "amount": 300.0, "date": "2024-03-16"},
     {"order_id": 4, "customer": "Diana", "amount": 400.0, "date": "2024-03-16"}],
]

for i, records in enumerate(files_data):
    filepath = os.path.join(sample_dir, f"orders_batch_{i+1}.json")
    with open(filepath, "w") as f:
        for record in records:
            f.write(json.dumps(record) + "\n")

print("✅ Created sample files for Auto Loader demo")
print(f"   Directory: {sample_dir}")
for f in os.listdir(sample_dir):
    print(f"   📄 {f}")

In [ ]:
# Auto Loader ingestion (using cloudFiles format)
# Note: This uses directory listing mode (works on Community Edition)

print("🔄 AUTO LOADER INGESTION")
print("=" * 60)

# Read with Auto Loader
try:
    df_stream = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", "/tmp/auto_loader_demo_schema")
        .option("cloudFiles.inferColumnTypes", "true")
        .load("/tmp/auto_loader_demo/"))
    
    # Write to Delta table
    spark.sql("DROP TABLE IF EXISTS techmart_dw.auto_loader_orders")
    
    (df_stream.writeStream
        .format("delta")
        .option("checkpointLocation", "/tmp/auto_loader_demo_checkpoint")
        .trigger(availableNow=True)
        .outputMode("append")
        .toTable("techmart_dw.auto_loader_orders"))
    
    # Verify
    count = spark.table("techmart_dw.auto_loader_orders").count()
    print(f"   ✅ Ingested {count} records via Auto Loader")
    spark.table("techmart_dw.auto_loader_orders").show()
    
except Exception as e:
    print(f"   ⚠️ Auto Loader demo: {e}")
    print("   (cloudFiles may need specific path format on Community Edition)")
    print("   In production, this works seamlessly with S3/ADLS/GCS paths.")

---
# 📗 Section 2: Schema Inference & Evolution

## Schema Handling Options

| Option | Behavior | Use When |
|--------|----------|----------|
| **Infer** | Auto-detect schema from files | First time, unknown schema |
| **Provide** | You specify the schema | Known, stable schema |
| **Evolve** | Auto-add new columns | Source adds columns over time |
| **Rescue** | Put unparseable data in _rescued_data column | Handling corrupt records |

## Schema Evolution Modes

```python
.option("cloudFiles.schemaEvolutionMode", "addNewColumns")  # Add new cols automatically
.option("cloudFiles.schemaEvolutionMode", "rescue")         # Put new cols in rescue column
.option("cloudFiles.schemaEvolutionMode", "failOnNewColumns") # Fail if schema changes
.option("cloudFiles.schemaEvolutionMode", "none")           # Ignore new columns
```

## The Rescue Column

When Auto Loader encounters data that doesn't match the schema:
```python
.option("cloudFiles.schemaEvolutionMode", "rescue")
# Creates a _rescued_data column with the unparseable fields as JSON
```

This prevents data loss while maintaining schema stability.

---
*Notebook 36 of the Databricks Data Engineering series*
*Study Order: 9*

---
# 📗 Section 3: Auto Loader Core API

## The cloudFiles Format

Auto Loader uses `spark.readStream.format("cloudFiles")` — the core API:

```python
df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")          # Source file format
    .option("cloudFiles.schemaLocation", "/path") # Where to store inferred schema
    .option("cloudFiles.inferColumnTypes", "true") # Infer types (not just strings)
    .load("/raw/orders/"))                         # Source directory
```

### Required Options

| Option | Description | Example |
|--------|-------------|---------|
| `cloudFiles.format` | Format of source files | `json`, `csv`, `parquet`, `avro`, `text` |
| `cloudFiles.schemaLocation` | Where to persist inferred schema | `/checkpoints/orders/schema` |

### Key Optional Options

| Option | Default | Description |
|--------|---------|-------------|
| `cloudFiles.inferColumnTypes` | `false` | Infer types (true) or keep as strings (false) |
| `cloudFiles.schemaHints` | None | Override types for specific columns |
| `cloudFiles.schemaEvolutionMode` | `addNewColumns` | How to handle new columns |
| `cloudFiles.maxFilesPerTrigger` | unlimited | Limit files per micro-batch |
| `cloudFiles.maxBytesPerTrigger` | unlimited | Limit bytes per micro-batch |
| `cloudFiles.useNotifications` | `false` | Use cloud events vs directory listing |
| `cloudFiles.includeExistingFiles` | `true` | Process files already in directory |
| `cloudFiles.pathGlobFilter` | None | Filter files by pattern (e.g., `*.json`) |

### File Format Options

```python
# JSON
.option("cloudFiles.format", "json")
.option("multiLine", "true")          # For multi-line JSON objects

# CSV
.option("cloudFiles.format", "csv")
.option("header", "true")
.option("delimiter", ",")
.option("quote", '"')

# Parquet (no extra options needed usually)
.option("cloudFiles.format", "parquet")

# Avro
.option("cloudFiles.format", "avro")

# Binary files (images, PDFs)
.option("cloudFiles.format", "binaryFile")
```

In [ ]:
# Auto Loader demo — set up sample files and ingest them
import json, os

# Create sample JSON files to ingest
sample_dir = "/tmp/autoloader_demo/orders"
os.makedirs(sample_dir, exist_ok=True)

# Batch 1: Initial files
batch1 = [
    {"order_id": 1, "customer_id": 42, "amount": 99.99, "status": "completed", "date": "2024-03-15"},
    {"order_id": 2, "customer_id": 17, "amount": 149.50, "status": "pending", "date": "2024-03-15"},
    {"order_id": 3, "customer_id": 55, "amount": 299.00, "status": "shipped", "date": "2024-03-15"},
]

with open(f"{sample_dir}/orders_2024-03-15_001.json", "w") as f:
    for record in batch1:
        f.write(json.dumps(record) + "\n")

# Batch 2: More files
batch2 = [
    {"order_id": 4, "customer_id": 42, "amount": 75.00, "status": "completed", "date": "2024-03-16"},
    {"order_id": 5, "customer_id": 88, "amount": 450.00, "status": "pending", "date": "2024-03-16"},
]

with open(f"{sample_dir}/orders_2024-03-16_001.json", "w") as f:
    for record in batch2:
        f.write(json.dumps(record) + "\n")

print("✅ Created sample files:")
for f in os.listdir(sample_dir):
    size = os.path.getsize(f"{sample_dir}/{f}")
    print(f"   {f} ({size} bytes)")
print(f"\nTotal: {len(batch1) + len(batch2)} records across 2 files")

In [ ]:
# Run Auto Loader to ingest the files
print("🔄 AUTO LOADER INGESTION")
print("=" * 60)

# Clean up any previous run
spark.sql("DROP TABLE IF EXISTS techmart_dw.autoloader_orders_demo")

try:
    # Read with Auto Loader
    df_stream = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", "/tmp/autoloader_demo/schema")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.includeExistingFiles", "true")
        .load("/tmp/autoloader_demo/orders/"))
    
    # Add metadata columns (standard practice)
    from pyspark.sql.functions import current_timestamp, input_file_name, lit
    df_with_meta = (df_stream
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_file", input_file_name()))
    
    # Write to Delta table (trigger once — process all available files)
    (df_with_meta.writeStream
        .format("delta")
        .option("checkpointLocation", "/tmp/autoloader_demo/checkpoint")
        .trigger(availableNow=True)
        .outputMode("append")
        .toTable("techmart_dw.autoloader_orders_demo"))
    
    # Verify
    result = spark.table("techmart_dw.autoloader_orders_demo")
    print(f"   ✅ Ingested {result.count()} records")
    result.show(truncate=False)
    
except Exception as e:
    print(f"   ⚠️ Auto Loader demo: {e}")
    print("   (Requires DBFS/cloud storage path in production)")
    print("   In production: .load('s3://bucket/raw/orders/')")

---
# 📗 Section 4: File Detection Modes

## Directory Listing vs File Notification

Auto Loader supports two modes for detecting new files:

### Mode 1: Directory Listing (Default — Works on Community Edition)

```
Auto Loader periodically lists the directory:
  /raw/orders/
    ├── 2024-03-15_001.json  ← Already processed (tracked in checkpoint)
    ├── 2024-03-15_002.json  ← Already processed
    └── 2024-03-16_001.json  ← NEW! Process this.
```

**How it works:**
1. Auto Loader lists all files in the directory
2. Compares against the list of already-processed files (stored in checkpoint)
3. Processes only new files
4. Updates the checkpoint

**Pros:** Simple setup, works everywhere
**Cons:** Slower for millions of files (listing is O(n))

### Mode 2: File Notification (Production — Requires Cloud Setup)

```
Cloud Storage → Event Notification → SQS/Event Grid/Pub/Sub → Auto Loader
                (file created event)
```

**How it works:**
1. Cloud storage sends an event when a new file is created
2. Auto Loader receives the event immediately
3. Processes the file within seconds

**Pros:** Near-instant detection, scales to billions of files
**Cons:** Requires cloud setup (Auto Loader creates the resources automatically)

### Switching Between Modes

```python
# Directory listing (default)
.option("cloudFiles.useNotifications", "false")

# File notification (production)
.option("cloudFiles.useNotifications", "true")
# Auto Loader automatically creates:
# - AWS: SQS queue + S3 event notifications
# - Azure: Event Grid subscription
# - GCP: Pub/Sub topic + GCS notifications
```

### When to Use Each

| Scenario | Mode | Why |
|----------|------|-----|
| Development/testing | Directory listing | Simple, no cloud setup |
| < 1M files total | Directory listing | Fast enough |
| > 1M files | File notification | Listing too slow |
| Low latency required (< 30s) | File notification | Events are instant |
| Community Edition | Directory listing | Only option available |

In [ ]:
# Demonstrate directory listing mode behavior
print("📁 DIRECTORY LISTING MODE — How Auto Loader Tracks Files")
print("=" * 60)

# Simulate what Auto Loader's checkpoint tracks
class AutoLoaderCheckpoint:
    """Simulates Auto Loader's file tracking mechanism."""
    
    def __init__(self, checkpoint_path):
        self.checkpoint_path = checkpoint_path
        self.processed_files = set()
        self.total_records = 0
    
    def get_new_files(self, all_files):
        """Find files not yet processed."""
        return [f for f in all_files if f not in self.processed_files]
    
    def mark_processed(self, files, records_count):
        """Update checkpoint after processing."""
        self.processed_files.update(files)
        self.total_records += records_count
    
    def status(self):
        return {
            "processed_files": len(self.processed_files),
            "total_records": self.total_records,
            "checkpoint": self.checkpoint_path
        }


# Simulate 3 runs of Auto Loader
checkpoint = AutoLoaderCheckpoint("/checkpoints/orders/")

# Run 1: 2 files exist
print("\n--- RUN 1 (initial) ---")
all_files = ["orders_2024-03-15_001.json", "orders_2024-03-15_002.json"]
new_files = checkpoint.get_new_files(all_files)
print(f"   Files in directory: {len(all_files)}")
print(f"   New files to process: {new_files}")
checkpoint.mark_processed(new_files, records_count=500)
print(f"   Checkpoint: {checkpoint.status()}")

# Run 2: 1 new file added
print("\n--- RUN 2 (1 new file) ---")
all_files.append("orders_2024-03-16_001.json")
new_files = checkpoint.get_new_files(all_files)
print(f"   Files in directory: {len(all_files)}")
print(f"   New files to process: {new_files}")
checkpoint.mark_processed(new_files, records_count=250)
print(f"   Checkpoint: {checkpoint.status()}")

# Run 3: No new files
print("\n--- RUN 3 (no new files) ---")
new_files = checkpoint.get_new_files(all_files)
print(f"   Files in directory: {len(all_files)}")
print(f"   New files to process: {new_files}")
print(f"   ⏭️ Nothing to process — Auto Loader skips this run")

print("\n💡 Key Insight:")
print("   Auto Loader NEVER reprocesses files (unless you reset the checkpoint).")
print("   This gives you exactly-once semantics for file ingestion.")


---
# 📗 Section 5: Schema Inference & Evolution

## The Schema Challenge

When files arrive from external systems, you often don't control the schema.
Auto Loader handles this with three strategies:

### Strategy 1: Schema Inference (First Run)

Auto Loader samples files to infer the schema:
```python
.option("cloudFiles.schemaLocation", "/checkpoints/orders/schema")
.option("cloudFiles.inferColumnTypes", "true")
```

The inferred schema is **persisted** to `schemaLocation` so it's reused on subsequent runs.

### Strategy 2: Schema Hints (Override Specific Columns)

When inference gets a type wrong, provide hints:
```python
.option("cloudFiles.schemaHints", "order_id INT, amount DECIMAL(12,2), order_date DATE")
```

### Strategy 3: Schema Evolution Modes

What happens when a NEW column appears in incoming files?

| Mode | Behavior | Use When |
|------|----------|----------|
| `addNewColumns` (default) | Auto-add new columns to schema | Source adds columns over time |
| `rescue` | Put unknown columns in `_rescued_data` JSON column | Strict schema, capture extras |
| `failOnNewColumns` | Fail the pipeline | Schema must never change |
| `none` | Ignore new columns | Don't care about new columns |

### The Rescue Column

When `schemaEvolutionMode = "rescue"`, any data that doesn't fit the schema
goes into a special `_rescued_data` column as JSON:

```
Source file has: {order_id: 1, amount: 99.99, new_field: "surprise!"}
Schema expects:  order_id, amount

Result:
  order_id = 1
  amount = 99.99
  _rescued_data = '{"new_field": "surprise!"}'  ← Captured, not lost!
```

This prevents data loss while maintaining schema stability.

In [ ]:
# Schema evolution simulation
print("🔄 SCHEMA EVOLUTION — How Auto Loader Handles Changes")
print("=" * 60)

# Simulate schema evolution scenarios
import json

# Original schema (inferred from first batch)
original_schema = {
    "order_id": "INT",
    "customer_id": "INT",
    "amount": "DECIMAL(12,2)",
    "status": "STRING",
    "order_date": "DATE"
}

# New batch has an extra column
new_batch_record = {
    "order_id": 6,
    "customer_id": 99,
    "amount": 199.99,
    "status": "completed",
    "order_date": "2024-03-17",
    "discount_code": "SAVE10",    # NEW column!
    "is_gift": True               # Another NEW column!
}

print("\n📌 Original schema:")
for col, dtype in original_schema.items():
    print(f"   {col}: {dtype}")

print("\n📌 New record has extra columns:")
print(f"   discount_code: 'SAVE10'  ← NEW")
print(f"   is_gift: True            ← NEW")

print("\n📊 Schema Evolution Mode Behavior:")
print("-" * 60)

modes = {
    "addNewColumns (default)": {
        "action": "Auto-add discount_code and is_gift to schema",
        "result": "All data captured, schema grows",
        "risk": "Schema drift — downstream consumers may break"
    },
    "rescue": {
        "action": "Keep original schema, put extras in _rescued_data",
        "result": "_rescued_data = '{"discount_code": "SAVE10", "is_gift": true}'",
        "risk": "Need to parse _rescued_data to access new fields"
    },
    "failOnNewColumns": {
        "action": "Pipeline FAILS with schema mismatch error",
        "result": "No data processed until schema is updated",
        "risk": "Pipeline downtime until manually resolved"
    },
    "none": {
        "action": "Silently ignore discount_code and is_gift",
        "result": "Data lost! New columns never captured",
        "risk": "Silent data loss — dangerous!"
    }
}

for mode, details in modes.items():
    print(f"\n   Mode: {mode}")
    print(f"   Action: {details['action']}")
    print(f"   Result: {details['result']}")
    print(f"   Risk: {details['risk']}")

print("\n💡 Recommendation:")
print("   • Use 'addNewColumns' for flexible pipelines")
print("   • Use 'rescue' when you need strict schema + capture extras")
print("   • NEVER use 'none' (silent data loss)")
print("   • Use 'failOnNewColumns' only for critical financial data")


---
# 📗 Section 6: Production Patterns

## Pattern 1: Auto Loader → Bronze → Silver Pipeline

The most common production pattern:

```python
# Step 1: Auto Loader reads files → Bronze (raw, append-only)
bronze_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/checkpoints/orders/schema")
    .option("cloudFiles.inferColumnTypes", "true")
    .load("s3://my-bucket/raw/orders/")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", input_file_name()))

(bronze_stream.writeStream
    .format("delta")
    .option("checkpointLocation", "/checkpoints/orders/bronze")
    .trigger(availableNow=True)  # Batch mode
    .outputMode("append")
    .toTable("bronze.raw_orders"))

# Step 2: Bronze → Silver (separate job, runs after bronze)
silver_df = (spark.table("bronze.raw_orders")
    .filter(col("order_id").isNotNull())
    .withColumn("amount", col("amount").cast("decimal(12,2)"))
    .withColumn("status", lower(trim(col("status")))))

(silver_df.write
    .format("delta")
    .mode("append")
    .saveAsTable("silver.clean_orders"))
```

## Pattern 2: Trigger Modes

| Trigger | Behavior | Use For |
|---------|----------|---------|
| `availableNow=True` | Process all available files, then stop | Scheduled batch jobs |
| `processingTime="5 minutes"` | Run every 5 minutes | Near-real-time |
| `once=True` | Process one micro-batch, then stop | Legacy (use availableNow) |
| No trigger | Continuous streaming | Real-time (always running) |

## Pattern 3: Handling Multiple File Formats

```python
# Different directories for different formats
orders_stream = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .load("s3://bucket/raw/orders/"))

products_stream = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .load("s3://bucket/raw/products/"))

events_stream = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .load("s3://bucket/raw/events/"))
```

## Pattern 4: Backfill (Reprocess Historical Files)

```python
# Reset checkpoint to reprocess all files from scratch
# WARNING: This will reprocess ALL files!
dbutils.fs.rm("/checkpoints/orders/", recurse=True)

# Or: process only files from a specific date
.option("cloudFiles.includeExistingFiles", "true")
.option("cloudFiles.maxFilesPerTrigger", 100)  # Process in batches
```

## Pattern 5: Monitoring Auto Loader

```python
# Check streaming query status
query = df.writeStream.start()
print(query.status)
print(query.lastProgress)

# Key metrics to monitor:
# - inputRowsPerSecond: How fast files are being read
# - processedRowsPerSecond: How fast records are processed
# - numInputRows: Records in last micro-batch
# - triggerExecution.durationMs: How long each trigger took
```

In [ ]:
# Production Auto Loader patterns
print("🏭 PRODUCTION AUTO LOADER PATTERNS")
print("=" * 60)

# Pattern: Trigger modes comparison
trigger_modes = {
    "availableNow=True": {
        "code": "trigger(availableNow=True)",
        "behavior": "Process ALL available files, then stop",
        "use_case": "Scheduled batch job (hourly/daily)",
        "cost": "Pay only for compute time",
        "example": "Nightly ETL: process all files from today"
    },
    "processingTime='5 minutes'": {
        "code": "trigger(processingTime='5 minutes')",
        "behavior": "Run every 5 minutes, process new files",
        "use_case": "Near-real-time (5-minute latency OK)",
        "cost": "Cluster runs continuously",
        "example": "Dashboard refresh every 5 minutes"
    },
    "Continuous (no trigger)": {
        "code": "# No .trigger() call",
        "behavior": "Always running, processes files as they arrive",
        "use_case": "Real-time (seconds latency)",
        "cost": "Cluster always running (most expensive)",
        "example": "Real-time fraud detection"
    }
}

for mode, details in trigger_modes.items():
    print(f"\n📌 {mode}")
    print(f"   Code: {details['code']}")
    print(f"   Behavior: {details['behavior']}")
    print(f"   Use case: {details['use_case']}")
    print(f"   Cost: {details['cost']}")
    print(f"   Example: {details['example']}")


---
# 📗 Section 7: Auto Loader vs Alternatives

## When to Use Auto Loader vs Other Approaches

| Approach | Best For | Limitations |
|----------|----------|-------------|
| **Auto Loader** | Files in cloud storage (S3/ADLS/GCS) | Files only, not databases |
| **Lakeflow Connect** | Database CDC (PostgreSQL, MySQL, Oracle) | Requires paid tier |
| **Kafka + Spark Streaming** | Real-time event streams | Complex setup |
| **Manual spark.read** | One-time loads, small files | No incremental tracking |
| **COPY INTO** | Bulk loading from cloud storage | No streaming, no schema evolution |

## Auto Loader vs COPY INTO

Both load files from cloud storage, but they're different:

| Aspect | Auto Loader | COPY INTO |
|--------|-------------|-----------|
| **Streaming** | ✅ Yes (streaming) | ❌ No (batch only) |
| **Incremental** | ✅ Automatic tracking | ✅ Tracks processed files |
| **Schema evolution** | ✅ Built-in | ❌ Manual |
| **Rescue column** | ✅ Yes | ❌ No |
| **Complexity** | Medium | Low |
| **Use for** | Continuous ingestion | One-time or scheduled loads |

```sql
-- COPY INTO (simpler, batch only)
COPY INTO bronze.orders
FROM 's3://bucket/raw/orders/'
FILEFORMAT = JSON
FORMAT_OPTIONS ('inferSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');
```

In [ ]:
# ============================================
# 🎯 YOUR TURN: Design an Auto Loader Pipeline
# ============================================
# Scenario: You're building a data platform for a logistics company.
# Files arrive in S3 every 15 minutes:
#   - /raw/shipments/*.json (shipment events)
#   - /raw/tracking/*.csv (GPS tracking updates)
#   - /raw/manifests/*.parquet (daily manifests)
#
# Design the Auto Loader configuration for each source:
# 1. What format option?
# 2. What trigger mode? (15-min latency OK for shipments, 1-min for tracking)
# 3. What schema evolution mode?
# 4. What expectations/quality checks?
# 5. Where does each land (Bronze table name)?
#
# Write your design below:


In [ ]:
# ============================================
# ✅ SOLUTION: Logistics Auto Loader Design
# ============================================
logistics_design = {
    "shipments": {
        "source": "s3://logistics/raw/shipments/*.json",
        "format": "json",
        "trigger": "availableNow=True (run every 15 min via Lakeflow Jobs)",
        "schema_evolution": "addNewColumns (carrier may add new fields)",
        "schema_hints": "shipment_id INT, weight_kg DECIMAL(8,2), ship_date DATE",
        "quality_checks": [
            "shipment_id IS NOT NULL (drop)",
            "weight_kg > 0 (drop)",
            "destination IS NOT NULL (warn)",
        ],
        "bronze_table": "bronze.raw_shipments",
        "checkpoint": "/checkpoints/shipments/",
    },
    "tracking": {
        "source": "s3://logistics/raw/tracking/*.csv",
        "format": "csv",
        "trigger": "processingTime='1 minute' (near-real-time for GPS)",
        "schema_evolution": "rescue (strict schema, capture extras)",
        "schema_hints": "device_id STRING, lat DOUBLE, lon DOUBLE, ts TIMESTAMP",
        "quality_checks": [
            "device_id IS NOT NULL (drop)",
            "lat BETWEEN -90 AND 90 (drop)",
            "lon BETWEEN -180 AND 180 (drop)",
        ],
        "bronze_table": "bronze.raw_tracking",
        "checkpoint": "/checkpoints/tracking/",
    },
    "manifests": {
        "source": "s3://logistics/raw/manifests/*.parquet",
        "format": "parquet",
        "trigger": "availableNow=True (daily batch)",
        "schema_evolution": "failOnNewColumns (manifests have fixed schema)",
        "schema_hints": "N/A (Parquet has embedded schema)",
        "quality_checks": [
            "manifest_id IS NOT NULL (fail)",
            "total_weight > 0 (drop)",
        ],
        "bronze_table": "bronze.raw_manifests",
        "checkpoint": "/checkpoints/manifests/",
    }
}

print("📋 LOGISTICS AUTO LOADER DESIGN")
print("=" * 60)
for source, config in logistics_design.items():
    print(f"\n📌 {source.upper()}")
    for key, value in config.items():
        if isinstance(value, list):
            print(f"   {key}:")
            for v in value:
                print(f"      • {v}")
        else:
            print(f"   {key}: {value}")


---
# 📗 Section 9: Auto Loader in DLT Pipelines

## Auto Loader + DLT = The Recommended Pattern

The most common production pattern combines Auto Loader with DLT:

```python
import dlt
from pyspark.sql.functions import current_timestamp, input_file_name

@dlt.table(
    name="bronze_orders",
    comment="Raw orders ingested via Auto Loader",
    table_properties={"quality": "bronze"}
)
def bronze_orders():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", "/checkpoints/orders/schema")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load("s3://my-bucket/raw/orders/")
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_file", input_file_name())
    )
```

## Auto Loader Performance Tuning

| Option | Purpose | Recommendation |
|--------|---------|----------------|
| `maxFilesPerTrigger` | Limit files per batch | Set for backfill scenarios |
| `maxBytesPerTrigger` | Limit bytes per batch | Prevents OOM on large files |
| `cloudFiles.backfillInterval` | Periodic full directory scan | Use for notification mode |
| `cloudFiles.numPartitions` | Parallelism for file listing | Match to cluster cores |

## Handling Corrupt Records

```python
# Use PERMISSIVE mode + rescue column
df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/checkpoints/schema")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .load("/raw/orders/"))

# Good records
good = df.filter("_corrupt_record IS NULL")

# Bad records -- investigate and fix
bad = df.filter("_corrupt_record IS NOT NULL")
```

In [ ]:
# Auto Loader patterns summary
print("Auto Loader Quick Reference")
print("=" * 60)

patterns = {
    "Basic JSON ingestion": """
spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/checkpoints/schema")
    .load("/raw/orders/")""",

    "CSV with header": """
spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", "/checkpoints/schema")
    .load("/raw/products/")""",

    "With schema hints": """
spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaHints", "order_id INT, amount DECIMAL(12,2)")
    .option("cloudFiles.schemaLocation", "/checkpoints/schema")
    .load("/raw/orders/")""",

    "Batch mode (process all, then stop)": """
(df.writeStream
    .format("delta")
    .option("checkpointLocation", "/checkpoints/orders")
    .trigger(availableNow=True)
    .toTable("bronze.orders"))""",

    "Near-real-time (every 5 min)": """
(df.writeStream
    .format("delta")
    .option("checkpointLocation", "/checkpoints/orders")
    .trigger(processingTime="5 minutes")
    .toTable("bronze.orders"))""",
}

for name, pattern in patterns.items():
    print(f"\n{name}:")
    print(pattern)


---
# 📗 Section 8: Common Issues & Troubleshooting

## Issue 1: Schema Mismatch Error

```
AnalysisException: Failed to merge fields 'amount' and 'amount'.
Failed to merge incompatible data types DoubleType and StringType
```

**Cause:** New files have a different type for an existing column.
**Fix:** Use `schemaHints` to enforce the correct type, or use `rescue` mode.

## Issue 2: Checkpoint Corruption

```
StreamingQueryException: Failed to read checkpoint
```

**Cause:** Checkpoint files corrupted (e.g., cluster killed mid-write).
**Fix:** Delete the checkpoint and restart (will reprocess all files).

```python
dbutils.fs.rm("/checkpoints/orders/", recurse=True)
```

## Issue 3: Files Not Being Detected

**Possible causes:**
- Files added before Auto Loader started (use `includeExistingFiles=true`)
- Wrong path (check with `dbutils.fs.ls("/raw/orders/")`)
- File format mismatch (CSV files in a JSON-configured loader)
- Permissions issue (check IAM/service principal access)

## Issue 4: Slow Ingestion (Too Many Small Files)

**Cause:** Thousands of tiny files → slow listing + many tasks.
**Fix:**
```python
.option("cloudFiles.maxFilesPerTrigger", 1000)  # Process in batches
.option("cloudFiles.maxBytesPerTrigger", "1g")  # Or limit by size
```

## Issue 5: Duplicate Records

**Cause:** Checkpoint reset or pipeline restarted without checkpoint.
**Fix:** Auto Loader is exactly-once IF you don't reset the checkpoint.
Use MERGE in Silver to handle any duplicates that slip through.

---
# 📗 Summary & Quick Reference

```
AUTO LOADER QUICK REFERENCE:

BASIC PATTERN:
  spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json|csv|parquet|avro")
    .option("cloudFiles.schemaLocation", "/checkpoints/name/schema")
    .option("cloudFiles.inferColumnTypes", "true")
    .load("/path/to/files/")

TRIGGER MODES:
  .trigger(availableNow=True)          # Batch: process all, then stop
  .trigger(processingTime="5 minutes") # Near-real-time
  # No trigger                         # Continuous streaming

SCHEMA EVOLUTION:
  addNewColumns (default)  # Auto-add new columns
  rescue                   # Capture extras in _rescued_data
  failOnNewColumns         # Strict: fail on any change
  none                     # Ignore new columns (dangerous!)

PRODUCTION CHECKLIST:
  ✅ Set schemaLocation (persist schema across restarts)
  ✅ Set checkpointLocation (exactly-once semantics)
  ✅ Add _ingested_at and _source_file metadata columns
  ✅ Use trigger(availableNow=True) for scheduled jobs
  ✅ Monitor with query.lastProgress
  ✅ Set up alerts for pipeline failures
```

---
*Notebook 09 of the Databricks Data Engineering series*
*Study Order: 9*